# Regularized Regression Project

Build a Ridge, Lasso, and ElasticNet models that predict the `price` column in the dataset on San Francisco Apartment rentals. Make sure to go through all the the relevant steps of the modelling workflow.

1. Use the model you built for the prior project as the basis for comparison. Does regularization improve fit?
2. Feel free to skip the EDA and checking of assumptions again
2. Engineer (or un-engineer previously) engineered Features as needed
3. Fit a Lasso, Ridge, and Elastic Net Regression using the features in your original model.
4. Once you are ready, fit your final model and report final model performance estimate by scoring on the test data. Report both test R-squared and MAE.
5. What happens to your error if you only model apartments <= 6000 in price... should we do this?

Advice:

1. Remember, regularization doesn't always help, but it can, especially if you let it choose features for you!

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score as r2, mean_absolute_error as mae, mean_squared_error as mse
from sklearn.model_selection import train_test_split

rentals_df = pd.read_csv("Data/sf_clean.csv") #.query("price <= 6000")

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district
0,6800,1600.0,2.0,2.0,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0
1,3500,550.0,1.0,1.0,(a) in-unit,(a) both,(c) multi,(b) protected,7.0
2,5100,1300.0,2.0,1.0,(a) in-unit,(a) both,(c) multi,(d) no parking,7.0
3,9000,3500.0,3.0,2.5,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0
4,3100,561.0,1.0,1.0,(c) no laundry,(a) both,(c) multi,(d) no parking,7.0


### Data Dictionary

1. Price: The price of the rental and our target variable
2. sqft: The area in square feet of the rental
3. beds: The number of bedrooms in the rental
4. bath: The number of bathrooms in the rental
5. laundry: Does the rental have a laundry machine inside the house, a shared laundry machine, or no laundry on site?
6. pets: Does the rental allow pets? Cats only, dogs only or both cats and dogs?
7. Housing type: Is the rental in a multi-unit building, a building with two units, or a stand alone house? 
8. Parking: Does the apartment off a parking space? No, protected in a garage, off-street in a parking lot, or valet service?
9. Hood district: Which part of San Francisco is the apartment located?

![image info](SFAR_map.png)

## Feature Engineering

In [2]:
district_map = {
    1.0: 'west',
    2.0: 'southwest',
    3.0: 'southwest',
    4.0: 'central',
    5.0: 'central',
    6.0: 'central',
    7.0: 'marina',       
    8.0: 'north_beach',  
    9.0: 'fidi_soma',    
    10.0: 'southwest',  
}

rentals_df['district_zone'] = rentals_df['hood_district'].map(district_map)

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone
0,6800,1600.0,2.0,2.0,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0,marina
1,3500,550.0,1.0,1.0,(a) in-unit,(a) both,(c) multi,(b) protected,7.0,marina
2,5100,1300.0,2.0,1.0,(a) in-unit,(a) both,(c) multi,(d) no parking,7.0,marina
3,9000,3500.0,3.0,2.5,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0,marina
4,3100,561.0,1.0,1.0,(c) no laundry,(a) both,(c) multi,(d) no parking,7.0,marina


In [3]:
# Collapse the three parallel categories — their slopes are identical
# across sqft, beds, and bath in every plot
rentals_df['is_valet'] = (rentals_df['parking'] == '(a) valet').astype(int)
rentals_df['valet_x_sqft'] = rentals_df['is_valet'] * rentals_df['sqft']
rentals_df['valet_x_beds'] = rentals_df['is_valet'] * rentals_df['beds']
rentals_df['valet_x_bath'] = rentals_df['is_valet'] * rentals_df['bath']

rentals_df['has_parking']   = (rentals_df['parking'] != '(d) no parking').astype(int)

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking
0,6800,1600.0,2.0,2.0,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
1,3500,550.0,1.0,1.0,(a) in-unit,(a) both,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
2,5100,1300.0,2.0,1.0,(a) in-unit,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0
3,9000,3500.0,3.0,2.5,(a) in-unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
4,3100,561.0,1.0,1.0,(c) no laundry,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0


In [4]:
laundry_map = {
    '(a) in-unit':    'in_unit',
    '(b) on-site':    'not_in_unit',
    '(c) no laundry': 'not_in_unit',
}

rentals_df['laundry'] = rentals_df['laundry'].map(laundry_map)
rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking
0,6800,1600.0,2.0,2.0,in_unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
1,3500,550.0,1.0,1.0,in_unit,(a) both,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
2,5100,1300.0,2.0,1.0,in_unit,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0
3,9000,3500.0,3.0,2.5,in_unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1
4,3100,561.0,1.0,1.0,not_in_unit,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0


In [5]:
# Single family homes price-per-bedroom/bathroom differently than multi-unit
rentals_df['is_single'] = (rentals_df['housing_type'] == '(a) single').astype(int)

rentals_df['single_x_beds'] = rentals_df['is_single'] * rentals_df['beds']
rentals_df['single_x_bath'] = rentals_df['is_single'] * rentals_df['bath']

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking,is_single,single_x_beds,single_x_bath
0,6800,1600.0,2.0,2.0,in_unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
1,3500,550.0,1.0,1.0,in_unit,(a) both,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
2,5100,1300.0,2.0,1.0,in_unit,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0
3,9000,3500.0,3.0,2.5,in_unit,(d) no pets,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
4,3100,561.0,1.0,1.0,not_in_unit,(a) both,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0


In [6]:
pet_map = {
    '(a) both': 'allows_dogs',
    '(b) dogs': 'allows_dogs',
    '(c) cats': 'no_dogs',
    '(d) no pets': 'no_dogs',
}

rentals_df['pets'] = rentals_df['pets'].map(pet_map)

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking,is_single,single_x_beds,single_x_bath
0,6800,1600.0,2.0,2.0,in_unit,no_dogs,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
1,3500,550.0,1.0,1.0,in_unit,allows_dogs,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
2,5100,1300.0,2.0,1.0,in_unit,allows_dogs,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0
3,9000,3500.0,3.0,2.5,in_unit,no_dogs,(c) multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
4,3100,561.0,1.0,1.0,not_in_unit,allows_dogs,(c) multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0


In [7]:
housing_map = {
    '(a) single': 'single',
    '(b) double': 'multi',
    '(c) multi':  'multi',
}

rentals_df['housing_type'] = rentals_df['housing_type'].map(housing_map)

rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,hood_district,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking,is_single,single_x_beds,single_x_bath
0,6800,1600.0,2.0,2.0,in_unit,no_dogs,multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
1,3500,550.0,1.0,1.0,in_unit,allows_dogs,multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
2,5100,1300.0,2.0,1.0,in_unit,allows_dogs,multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0
3,9000,3500.0,3.0,2.5,in_unit,no_dogs,multi,(b) protected,7.0,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
4,3100,561.0,1.0,1.0,not_in_unit,allows_dogs,multi,(d) no parking,7.0,marina,0,0.0,0.0,0.0,0,0,0.0,0.0


In [8]:
rentals_df.drop(columns=['hood_district'], inplace = True)
rentals_df.head()

,price,sqft,beds,bath,laundry,pets,housing_type,parking,district_zone,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking,is_single,single_x_beds,single_x_bath
0,6800,1600.0,2.0,2.0,in_unit,no_dogs,multi,(b) protected,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
1,3500,550.0,1.0,1.0,in_unit,allows_dogs,multi,(b) protected,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
2,5100,1300.0,2.0,1.0,in_unit,allows_dogs,multi,(d) no parking,marina,0,0.0,0.0,0.0,0,0,0.0,0.0
3,9000,3500.0,3.0,2.5,in_unit,no_dogs,multi,(b) protected,marina,0,0.0,0.0,0.0,1,0,0.0,0.0
4,3100,561.0,1.0,1.0,not_in_unit,allows_dogs,multi,(d) no parking,marina,0,0.0,0.0,0.0,0,0,0.0,0.0


In [9]:
rentals_df = pd.get_dummies(rentals_df, drop_first=True, dtype=int)
rentals_df.head()

,price,sqft,beds,bath,is_valet,valet_x_sqft,valet_x_beds,valet_x_bath,has_parking,is_single,...,pets_no_dogs,housing_type_single,parking_(b) protected,parking_(c) off-street,parking_(d) no parking,district_zone_fidi_soma,district_zone_marina,district_zone_north_beach,district_zone_southwest,district_zone_west
0,6800,1600.0,2.0,2.0,0,0.0,0.0,0.0,1,0,...,1,0,1,0,0,0,1,0,0,0
1,3500,550.0,1.0,1.0,0,0.0,0.0,0.0,1,0,...,0,0,1,0,0,0,1,0,0,0
2,5100,1300.0,2.0,1.0,0,0.0,0.0,0.0,0,0,...,0,0,0,0,1,0,1,0,0,0
3,9000,3500.0,3.0,2.5,0,0.0,0.0,0.0,1,0,...,1,0,1,0,0,0,1,0,0,0
4,3100,561.0,1.0,1.0,0,0.0,0.0,0.0,0,0,...,0,0,0,0,1,0,1,0,0,0


## Data Splitting

In [15]:
X = rentals_df.drop(columns=['price'])
y = np.log(rentals_df['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2023)

## Scaling Data

In [16]:
from sklearn.preprocessing import StandardScaler

std = StandardScaler()
X_tr = std.fit_transform(X_train.values)
X_tst = std.transform(X_test.values)

## Ridge

In [17]:
from sklearn.linear_model import RidgeCV

n_alphas = 200
alphas = 10 ** np.linspace(-3, 3, n_alphas)

ridge_model = RidgeCV(alphas = alphas, cv = 5)
ridge_model.fit(X_tr, y_train)

print(f'Alpha: {ridge_model.alpha_}')
print(f'Train R2: {ridge_model.score(X_tr, y_train)}')
print(f"Training MAE: {mae(np.exp(y_train), np.exp(ridge_model.predict(X_tr)))}")

Alpha: 20.49074689815846
Train R2: 0.8165653345821493
Training MAE: 479.7786833898875


## Lasso

In [18]:
from sklearn.linear_model import LassoCV

n_alphas = 200
alphas = 10 ** np.linspace(-3, 3, n_alphas)

lasso_model = LassoCV(alphas = alphas, cv = 5)
lasso_model.fit(X_tr, y_train)

print(f'Alpha: {lasso_model.alpha_}')
print(f'Train R2: {lasso_model.score(X_tr, y_train)}')
print(f"Training MAE: {mae(np.exp(y_train), np.exp(lasso_model.predict(X_tr)))}")

Alpha: 0.001
Train R2: 0.8165440424231053
Training MAE: 482.2971236632148


## ENET

In [19]:
from sklearn.linear_model import ElasticNetCV

n_alphas = 200
alphas = 10 ** np.linspace(-3, 3, n_alphas)
l1_ratios = [x * .01 for x in range(1, 11)]

enet_model = ElasticNetCV(alphas = alphas, l1_ratio = l1_ratios, cv = 5)
enet_model = enet_model.fit(X_tr, y_train)

print(f'Tuned Alpha: {enet_model.alpha_}')
print(f'Tuned Lambda: {enet_model.l1_ratio_}')
print(f'Training Score: {enet_model.score(X_tr, y_train)}')
print(f"Training MAE: {mae(np.exp(y_train), np.exp(enet_model.predict(X_tr)))}")

Tuned Alpha: 0.026126752255633278
Tuned Lambda: 0.01
Training Score: 0.8163977194644326
Training MAE: 479.9827194035334


## Final Model Test

In [20]:
test_preds = ridge_model.predict(X_tst)

print(f'Test R2: {ridge_model.score(X_tst, y_test)}')
print(f'Test MAE: {round(mae(np.exp(y_test), np.exp(test_preds)), 3)}')

Test R2: 0.7786315347841066
Test MAE: 468.997
